# EdgeVision — full competition training and reliable submission

**Candidate:** YOLO11s, 640px, full training up to 50 epochs, ONNX CPU and TensorRT T4 FP32. This is a **competition experiment**, not a verified first-place result. Detection quality and inference speed must both be measured.

**Changes:** deterministic FP32 TensorRT starting point; diagnostic export parity check that tolerates small backend numerical differences while catching material mismatches; both `/kaggle/working/edgevision/submission.csv` and `/kaggle/working/submission.csv` are generated and checked.

**Usage:** Kaggle GPU T4 x2, Internet on, EdgeVision competition input attached. Run all cells in order. If a cell fails, stop at the first error. No code can guarantee error-free execution across changing Kaggle environments.

**Fair use:** train only on train labels, tune on validation, never use test-dev annotations. Report actual measured ONNX CPU / TensorRT T4 timings. Avoid publishing restricted frames or labels.


## 1. Install the libraries
Keep the platform's existing PyTorch/CUDA installation. Upgrade Ultralytics itself; install the remaining dependencies without requesting upgrades to the platform's existing plotting and data libraries. TensorRT's CUDA 12 package avoids accidentally installing CUDA 13 wheels on a CUDA 12 notebook. The TensorRT 10 series supports the T4 workflow used by this guide. If installation asks for a session restart, restart and run all cells again.

In [ ]:
%pip install -q -U ultralytics

In [ ]:
%pip install -q "onnx==1.20.0" "protobuf>=4.25.1,<6" "onnxruntime>=1.20" "onnxslim>=0.1.82" pandas matplotlib pyyaml tqdm

In [ ]:
%pip install -q "tensorrt-cu12>=10.0,<11"

## 2. Import the tools

In [ ]:
from pathlib import Path
from collections import defaultdict
import json
import os
import platform
import time
import shutil
import importlib.metadata as metadata

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import torch
import onnx
import onnxruntime as ort
import tensorrt as trt
from tqdm.auto import tqdm
from ultralytics import YOLO
from ultralytics.data.augment import LetterBox
from ultralytics.utils import ops, nms
from ultralytics.utils.metrics import ap_per_class

## 3. Candidate configuration

This version tries YOLO11s/640 with full training and TensorRT FP16. Both accuracy and speed matter. Evaluate both deployments on validation before choosing a final entry; a faster YOLO11n/320 configuration can outperform this larger model under the competition metric.


In [ ]:
MODE = "FULL"                         # Start with "SMOKE"; change to "FULL" for the final candidate
MODEL_NAME = "yolo11s.pt"              # Guide's optional reference model; not required by the competition
IMGSZ = 640                            # Reference baseline; compare 640 as a separate experiment
EPOCHS = 1 if MODE == "SMOKE" else 50
TRAIN_LIMIT = 128 if MODE == "SMOKE" else None
VAL_LIMIT = 64 if MODE == "SMOKE" else None
BATCH_SIZE = 8                         # Training batch; inference/benchmark batch remains 1
WORKERS = 2
PATIENCE = 12                          # Early stopping within the FULL epoch budget
TRT_FP16 = False                       # Start with guide-aligned FP32; test FP16 as a separate experiment
CONF = 0.001
DIAGNOSTIC_CONF = 0.25
NMS_IOU = 0.7
MAX_DET = 300
CPU_THREADS = 2                        # Required for comparability with the guide's CPU reference
SEED = 42
DATA_ROOT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path("/content/edgevision-data")
WORK = Path("/kaggle/working/edgevision") if Path("/kaggle/working").exists() else Path("/content/edgevision")
WORK.mkdir(parents=True, exist_ok=True)
assert MODE in {"SMOKE", "FULL"}
assert CPU_THREADS == 2, "Keep two CPU threads for comparability with the competition reference."
TEMP_ROOT = Path("/kaggle/temp/edgevision") if Path("/kaggle/working").exists() else Path("/content/edgevision_temp")
DATASET_WORK = TEMP_ROOT / f"yolo_data_{MODE.lower()}"
TRAIN_NAME = f"train_{Path(MODEL_NAME).stem}_{IMGSZ}_{MODE.lower()}_{'fp16' if TRT_FP16 else 'fp32'}"

### Check the GPU before doing expensive work
Choose GPU T4 on Kaggle. A different GPU can run this tutorial, but it does not satisfy the competition's T4 timing requirement. The notebook stops instead of inventing TensorRT results.

In [ ]:
assert torch.cuda.is_available(), "Enable GPU T4, then restart the notebook."
GPU_NAME = torch.cuda.get_device_name(0)
assert "T4" in GPU_NAME, f"The competition requires NVIDIA T4; found {GPU_NAME}."
assert int(torch.version.cuda.split(".")[0]) == 12, "Use a CUDA 12 PyTorch runtime with tensorrt-cu12."
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(CPU_THREADS)
print("GPU:", GPU_NAME, "| PyTorch CUDA:", torch.version.cuda, "| TensorRT:", trt.__version__)

## 4. Locate the extracted competition folders
Kaggle may add an extra archive or dataset directory. Locate the original split folders rather than assuming a fixed mount path. One shallow directory walk skips image and annotation contents, so discovery does not inspect more than 33,000 individual frames. Colab users should extract their archives under `DATA_ROOT` first.

In [ ]:
split_names = {"VisDrone2019-VID-train", "VisDrone2019-VID-val", "VisDrone2019-VID-test-dev"}
split_roots = defaultdict(list)
sample_paths = []
for current, directories, files in os.walk(DATA_ROOT):
    folder = Path(current)
    if folder.name in split_names and "sequences" in directories:
        split_roots[folder.name].append(folder)
    if "sample_submission.csv" in files:
        sample_paths.append(folder / "sample_submission.csv")
    directories[:] = [name for name in directories if name not in {"sequences", "annotations"}]

### Select the three split roots
Extra archive wrapper directories are supported; ambiguous duplicate datasets produce a clear error.

In [ ]:
def find_split(split_name):
    candidates = split_roots[split_name]
    assert len(candidates) == 1, f"Expected one {split_name} folder under {DATA_ROOT}; found {candidates}"
    return candidates[0]

train_source = find_split("VisDrone2019-VID-train")
val_source = find_split("VisDrone2019-VID-val")
test_source = find_split("VisDrone2019-VID-test-dev")
print("Train:", train_source, "\nValidation:", val_source, "\nTest:", test_source)

### Use the official sample submission as the test manifest
Every sample frame must appear exactly once, in the same order. Never benchmark only a small test subset and call it a complete submission.

In [ ]:
SUBMISSION_COLUMNS = [
    "frame_id", "onnx_cpu_detections_json", "onnx_cpu_latency_ms", "onnx_cpu_throughput_fps",
    "tensorrt_t4_detections_json", "tensorrt_t4_latency_ms", "tensorrt_t4_throughput_fps",
]
assert len(sample_paths) == 1, f"Attach the competition sample_submission.csv; found {sample_paths}"
sample = pd.read_csv(sample_paths[0], dtype={"frame_id": str})
assert list(sample.columns) == SUBMISSION_COLUMNS
assert len(sample) == 6635 and sample.frame_id.is_unique
test_ids = sample.frame_id.tolist()
test_paths = [test_source / "sequences" / frame_id for frame_id in test_ids]
assert all(p.is_file() for p in test_paths), "Some sample submission frames are missing."
assert not (test_source / "annotations").exists(), "Participant test folder must contain images only."
print(f"Ready: {len(test_paths):,} test frames")

## 5. Convert VisDrone video annotations to YOLO labels
Each sequence has a text file. The ten values in a row are frame number, track ID, left, top, width, height, score, category, truncation, and occlusion. Original categories **1–10** become YOLO classes **0–9**. Ignore category 0, category 11, and rows with score 0. Clip boxes to image boundaries. This tutorial's validation excludes these ignored objects; its AP is a useful diagnostic rather than a reproduction of the official VisDrone ignore-region evaluator.

In [ ]:
CLASS_NAMES = ["pedestrian", "people", "bicycle", "car", "van", "truck",
               "tricycle", "awning-tricycle", "bus", "motor"]

def read_video_annotations(annotation_path):
    by_frame = defaultdict(list)
    for line in annotation_path.read_text().splitlines():
        if not line.strip():
            continue
        row = [float(value) for value in line.split(",")]
        assert len(row) == 10, f"Invalid VID annotation: {annotation_path}"
        frame, _, left, top, width, height, score, category, _, _ = row
        if score > 0 and int(category) in range(1, 11):
            by_frame[int(frame)].append([int(category) - 1, left, top, left + width, top + height])
    return by_frame

In [ ]:
def clip_box(box, width, height):
    x1, y1, x2, y2 = box
    return [float(np.clip(x1, 0, width)), float(np.clip(y1, 0, height)),
            float(np.clip(x2, 0, width)), float(np.clip(y2, 0, height))]

def yolo_label(obj, width, height):
    class_id, *box = obj
    x1, y1, x2, y2 = clip_box(box, width, height)
    if x2 <= x1 or y2 <= y1:
        return None
    return f"{class_id} {(x1+x2)/(2*width):.8f} {(y1+y2)/(2*height):.8f} {(x2-x1)/width:.8f} {(y2-y1)/height:.8f}"

### Select frames spread across the split
Evenly spaced frames avoid taking only the first video. FULL uses every frame.

In [ ]:
def select_frames(source, limit):
    images = sorted((source / "sequences").glob("*/*.jpg"))
    if limit is not None and len(images) > limit:
        indices = np.linspace(0, len(images) - 1, limit, dtype=int)
        images = [images[i] for i in indices]
    return images

train_images = select_frames(train_source, TRAIN_LIMIT)
val_images = select_frames(val_source, VAL_LIMIT)
print(f"Training: {len(train_images):,}; validation: {len(val_images):,}; test: {len(test_paths):,}")

### Prepare one split
Links save disk space on Kaggle; copying is the fallback. Prepared images and labels stay in temporary storage outside the published output folder. Validation ground truth is also retained in original pixel coordinates for the deployment comparison.

In [ ]:
def prepare_split(source, image_paths, split):
    image_dir = DATASET_WORK / "images" / split
    label_dir = DATASET_WORK / "labels" / split
    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)
    annotations = {}
    ground_truth = {}
    selected_paths = []
    for image_path in tqdm(image_paths, desc=f"Prepare {split}"):
        sequence = image_path.parent.name
        if sequence not in annotations:
            annotations[sequence] = read_video_annotations(source / "annotations" / f"{sequence}.txt")
        image = cv2.imread(str(image_path))
        assert image is not None, f"Cannot decode {image_path}"
        height, width = image.shape[:2]
        objects = annotations[sequence].get(int(image_path.stem), [])
        lines, clipped = [], []
        for obj in objects:
            label = yolo_label(obj, width, height)
            if label is not None:
                lines.append(label)
                clipped.append([obj[0], *clip_box(obj[1:], width, height)])
        filename = f"{sequence}_{image_path.name}"
        destination = image_dir / filename
        if not destination.exists():
            try:
                destination.symlink_to(image_path.resolve())
            except OSError:
                shutil.copy2(image_path, destination)
        (label_dir / Path(filename).with_suffix(".txt")).write_text("\n".join(lines))
        selected_paths.append(str(destination.absolute()))
        ground_truth[str(image_path)] = np.asarray(clipped, dtype=np.float32).reshape(-1, 5)
    (DATASET_WORK / f"{split}.txt").write_text("\n".join(selected_paths))
    return ground_truth

In [ ]:
train_ground_truth = prepare_split(train_source, train_images, "train")

In [ ]:
val_ground_truth = prepare_split(val_source, val_images, "val")

### Write the dataset configuration

In [ ]:
data_yaml = WORK / "visdrone.yaml"
data_yaml.write_text(yaml.safe_dump({
    "path": str(DATASET_WORK), "train": "train.txt", "val": "val.txt",
    "names": dict(enumerate(CLASS_NAMES)),
}, sort_keys=False))
print(data_yaml.read_text())

## 6. Train YOLO11s on the full allowed training split

50 epochs is a maximum; early stopping may finish sooner. The training cell creates a new checkpoint for this configuration. If GPU quota is tight, set 30 epochs and 10 patience first.


In [ ]:
trainer = YOLO(MODEL_NAME)
training = trainer.train(
    data=str(data_yaml), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH_SIZE, device=0,
    workers=WORKERS, seed=SEED, deterministic=True, project=str(WORK / "runs"),
    name=TRAIN_NAME, exist_ok=True, plots=False, cache=False,
    patience=PATIENCE, cos_lr=True, close_mosaic=10, amp=True,
)


### Load the saved best checkpoint

In [ ]:
best_weights = Path(trainer.trainer.best)
assert best_weights.is_file()
trained_model = YOLO(str(best_weights))
assert len(trained_model.names) == 10
print("Best checkpoint:", best_weights)

### Evaluate the PyTorch checkpoint on validation images
AP measures ranking across confidence thresholds; diagnostic F1 below uses precision and recall returned by this validator. Image plots are disabled because they would copy restricted dataset frames into published notebook outputs. We generate deployment confusion matrices from counts later.

In [ ]:
pytorch_metrics = trained_model.val(
    data=str(data_yaml), split="val", imgsz=IMGSZ, batch=1, device=0,
    conf=CONF, iou=NMS_IOU, max_det=MAX_DET, plots=False,
    project=str(WORK / "runs"), name="pytorch_validation", exist_ok=True,
)
print({"AP50": float(pytorch_metrics.box.map50), "AP50_95": float(pytorch_metrics.box.map),
       "precision": float(pytorch_metrics.box.mp), "recall": float(pytorch_metrics.box.mr)})

## 7. Export ONNX
This reference decoder expects the YOLO11 detection head's raw `[1, 4 + classes, anchors]` output, with NMS performed afterward. If you choose a different model architecture, adapt its output decoder and verify both deployments again.

In [ ]:
onnx_path = Path(trained_model.export(
    format="onnx", imgsz=IMGSZ, batch=1, dynamic=False, simplify=True,
    opset=17, nms=False, device="cpu",
))
onnx.checker.check_model(onnx.load(str(onnx_path)))
print("Validated ONNX:", onnx_path)

## 8. Export TensorRT on this T4

The engine uses the same trained weights and fixed batch size 1. Keep `TRT_FP16=False` for the first guide-aligned FP32 baseline. After selecting a resolution, set `TRT_FP16=True` as a separate speed/precision experiment, regenerate the TensorRT engine, and rerun parity and validation. Quantization/precision changes can alter borderline scores and NMS decisions, so keep FP16 only if its measured accuracy-speed trade-off is better.

In [ ]:
engine_path = Path(trained_model.export(
    format="engine", imgsz=IMGSZ, batch=1, dynamic=False,
    nms=False, device=0, workspace=2, half=TRT_FP16,
))
assert engine_path.is_file()
print("TensorRT engine:", engine_path, "| FP16:", TRT_FP16)


## 9. Load ONNX explicitly on CPU
The provider assertion prevents accidental GPU execution in a notebook that also has CUDA.

In [ ]:
session_options = ort.SessionOptions()
session_options.intra_op_num_threads = CPU_THREADS
session_options.inter_op_num_threads = 1
onnx_session = ort.InferenceSession(str(onnx_path), sess_options=session_options,
                                    providers=["CPUExecutionProvider"])
assert onnx_session.get_providers() == ["CPUExecutionProvider"]
onnx_input_name = onnx_session.get_inputs()[0].name
print("ONNX providers:", onnx_session.get_providers())

### Prepare one image for ONNX
The input is a decoded OpenCV BGR image. Letterboxing keeps the aspect ratio; RGB channels are normalized to 0–1 and arranged as NCHW.

In [ ]:
letterbox = LetterBox(new_shape=(IMGSZ, IMGSZ), auto=False, stride=32)

def onnx_input(image):
    resized = letterbox(image=image)
    rgb = resized[:, :, ::-1].transpose(2, 0, 1)
    return np.ascontiguousarray(rgb[None], dtype=np.float32) / 255.0

### Decode and suppress duplicate ONNX detections
Output rows are `[x1, y1, x2, y2, confidence, zero_based_class]` in original image pixels.

In [ ]:
def predict_onnx(image):
    tensor = onnx_input(image)
    raw = onnx_session.run(None, {onnx_input_name: tensor})[0]
    assert raw.ndim == 3 and raw.shape[1] == 14, f"Adapt decoder for this model output: {raw.shape}"
    detections = nms.non_max_suppression(
        torch.from_numpy(raw), conf_thres=CONF, iou_thres=NMS_IOU,
        max_det=MAX_DET, nc=10,
    )[0]
    if len(detections):
        detections[:, :4] = ops.scale_boxes((IMGSZ, IMGSZ), detections[:, :4], image.shape[:2])
    return detections.cpu().numpy()

### Load TensorRT and define its prediction function

In [ ]:
trt_model = YOLO(str(engine_path), task="detect")

def predict_tensorrt(image):
    result = trt_model.predict(
        source=image, imgsz=IMGSZ, conf=CONF, iou=NMS_IOU,
        max_det=MAX_DET, device=0, rect=False, verbose=False,
    )[0]
    return result.boxes.data.detach().cpu().numpy()

## 10. Warm up both deployments
Warmup includes backend loading and CUDA initialization. None of this is counted as inference latency.

In [ ]:
warm_image = cv2.imread(str(test_paths[0]))
assert warm_image is not None, "Test frame could not be decoded"

for _ in range(10):
    predict_onnx(warm_image)
    predict_tensorrt(warm_image)
torch.cuda.synchronize()

trt_backend = trt_model.predictor.model
assert trt_backend.format == "engine", "TensorRT engine backend was not loaded"
assert trt_backend.device.type == "cuda", "TensorRT is not running on GPU"
print("ONNX provider:", onnx_session.get_providers())
print("TensorRT engine:", engine_path)
print("TensorRT backend fp16 flag:", trt_backend.fp16)
print("Both deployments loaded and warmed up successfully")


### Export parity diagnostic
This check verifies identical preprocessing and looks for **material** output mismatches.
Small floating-point differences between ONNX and TensorRT kernels are expected.
Validation AP and real runtime speed below are the deciding metrics, not exact bitwise agreement.


In [ ]:
parity_input = onnx_input(warm_image)
wrapper_input = trt_model.predictor.preprocess([warm_image])
np.testing.assert_allclose(
    wrapper_input.float().cpu().numpy(),
    parity_input,
    rtol=0,
    atol=1e-6,
    err_msg="ONNX and TensorRT preprocessing must match",
)

onnx_raw = onnx_session.run(None, {onnx_input_name: parity_input})[0]
with torch.inference_mode():
    trt_raw = trt_model.predictor.model(wrapper_input)
if isinstance(trt_raw, (list, tuple)):
    trt_raw = trt_raw[0]
trt_raw = trt_raw.detach().float().cpu().numpy()

assert onnx_raw.shape == trt_raw.shape, "Deployment output shapes differ"
assert onnx_raw.ndim == 3 and onnx_raw.shape[1] == 14, "Unexpected YOLO detection output"
assert np.isfinite(onnx_raw).all() and np.isfinite(trt_raw).all(), "Non-finite export output"

differences = np.abs(onnx_raw - trt_raw)
normalized_mean_error = float(differences.mean() / max(np.abs(onnx_raw).mean(), 1e-6))
p999_coordinate_error = float(np.percentile(differences[:, :4], 99.9))
max_coordinate_error = float(differences[:, :4].max())
print(f"Normalized mean raw difference: {normalized_mean_error:.4%}")
print(f"99.9th percentile coordinate difference: {p999_coordinate_error:.4f}")
print(f"Maximum coordinate difference: {max_coordinate_error:.4f}")

# Raw engine kernels need not be bit-for-bit identical; the two detectors are
# evaluated independently on validation images below. Reject broad mismatches.
assert normalized_mean_error < 0.02, "Material ONNX/TensorRT mismatch: investigate exports"
assert p999_coordinate_error < 4.0, "Large raw coordinate mismatch: investigate exports"
print("Export consistency diagnostics passed. Validate both runtimes on labeled data next.")


## 11. Measure one frame
The timer covers preprocessing, model execution, NMS, scaling, and returning CPU detections. Decode images before the timer. TensorRT includes input/output transfer and synchronizes CUDA on both sides of the timer. JSON serialization, image reading, model loading and warmup are excluded. Use the same policy for every entry.

In [ ]:
def timed_prediction(predict, image, use_cuda):
    if use_cuda:
        torch.cuda.synchronize()
    start = time.perf_counter_ns()
    detections = predict(image)
    if use_cuda:
        torch.cuda.synchronize()
    latency_ms = (time.perf_counter_ns() - start) / 1_000_000
    assert np.isfinite(latency_ms) and latency_ms > 0
    return detections, latency_ms

runtimes = {
    "onnx_cpu": (predict_onnx, False),
    "tensorrt_t4": (predict_tensorrt, True),
}

## 12. Compare detection accuracy on validation labels
For confidence 0.25 and IoU 0.50: precision = TP/(TP+FP), recall = TP/(TP+FN), F1 is their harmonic mean. Detection accuracy here means **TP/(TP+FP+FN)**; object detection has no meaningful count of background true negatives. Confusion matrices include a background row/column for missed and extra detections. AP uses confidence-ranked class-correct matches at IoUs 0.50 through 0.95.

In [ ]:
def box_iou_numpy(boxes1, boxes2):
    left_top = np.maximum(boxes1[:, None, :2], boxes2[None, :, :2])
    right_bottom = np.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])
    intersection = np.maximum(right_bottom - left_top, 0).prod(axis=2)
    area1 = np.maximum(boxes1[:, 2:] - boxes1[:, :2], 0).prod(axis=1)
    area2 = np.maximum(boxes2[:, 2:] - boxes2[:, :2], 0).prod(axis=1)
    return intersection / np.maximum(area1[:, None] + area2[None, :] - intersection, 1e-9)

In [ ]:
def match_detections(detections, ground_truth, threshold, same_class=True):
    overlaps = box_iou_numpy(detections[:, :4], ground_truth[:, 1:])
    used_ground_truth, matches = set(), []
    for prediction in np.argsort(-detections[:, 4]):
        candidates = [g for g in range(len(ground_truth)) if g not in used_ground_truth
                      and overlaps[prediction, g] >= threshold
                      and (not same_class or int(detections[prediction, 5]) == int(ground_truth[g, 0]))]
        if candidates:
            target = max(candidates, key=lambda g: overlaps[prediction, g])
            used_ground_truth.add(target)
            matches.append((int(prediction), target))
    return matches

In [ ]:
def validation_statistics(detections, ground_truth):
    correct = np.zeros((len(detections), 10), dtype=bool)
    for column, threshold in enumerate(np.linspace(0.50, 0.95, 10)):
        for prediction, _ in match_detections(detections, ground_truth, threshold):
            correct[prediction, column] = True
    visible = detections[detections[:, 4] >= DIAGNOSTIC_CONF]
    class_matches = match_detections(visible, ground_truth, 0.50)
    tp = len(class_matches)
    counts = np.array([tp, len(visible) - tp, len(ground_truth) - tp])
    confusion = np.zeros((11, 11), dtype=int)  # Rows true; columns predicted
    spatial_matches = match_detections(visible, ground_truth, 0.50, same_class=False)
    matched_predictions = {p for p, _ in spatial_matches}
    matched_targets = {g for _, g in spatial_matches}
    for prediction, target in spatial_matches:
        confusion[int(ground_truth[target, 0]), int(visible[prediction, 5])] += 1
    for prediction in set(range(len(visible))) - matched_predictions:
        confusion[10, int(visible[prediction, 5])] += 1
    for target in set(range(len(ground_truth))) - matched_targets:
        confusion[int(ground_truth[target, 0]), 10] += 1
    return correct, counts, confusion

### Collect validation predictions for one runtime

In [ ]:
def evaluate_runtime(name, predict, use_cuda):
    correct_parts, confidences, classes, targets, latencies = [], [], [], [], []
    counts = np.zeros(3, dtype=int)
    confusion = np.zeros((11, 11), dtype=int)
    for image_path in tqdm(val_images, desc=f"Validate {name}"):
        image = cv2.imread(str(image_path))
        assert image is not None
        detections, latency = timed_prediction(predict, image, use_cuda)
        truth = val_ground_truth[str(image_path)]
        correct, frame_counts, frame_confusion = validation_statistics(detections, truth)
        correct_parts.append(correct)
        confidences.extend(detections[:, 4].tolist())
        classes.extend(detections[:, 5].tolist())
        targets.extend(truth[:, 0].tolist())
        counts += frame_counts
        confusion += frame_confusion
        latencies.append(latency)
    return dict(name=name, correct=np.concatenate(correct_parts), confidence=np.array(confidences),
                classes=np.array(classes), targets=np.array(targets), counts=counts,
                confusion=confusion, latencies=np.array(latencies))

In [ ]:
validation_outputs = {}
for name, (predict, use_cuda) in runtimes.items():
    validation_outputs[name] = evaluate_runtime(name, predict, use_cuda)

### Summarize accuracy and speed
Validation throughput is number of frames divided by total timed processing seconds. It has the same timer scope as latency. This guide does not count image disk loading as inference time.

In [ ]:
def summarize_validation(output):
    tp, fp, fn = output["counts"]
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    ap = np.zeros((1, 10))
    if len(output["confidence"]) and len(output["targets"]):
        ap = ap_per_class(output["correct"], output["confidence"], output["classes"],
                          output["targets"], plot=False)[5]
    confidence = output["confidence"][output["confidence"] >= DIAGNOSTIC_CONF]
    return {
        "runtime": output["name"], "validation_frames": len(val_images),
        "AP50": float(ap[:, 0].mean()) if ap.size else 0.0,
        "AP50_95": float(ap.mean()) if ap.size else 0.0,
        "precision": precision, "recall": recall,
        "f1_score": 2 * precision * recall / max(precision + recall, 1e-12),
        "detection_accuracy": tp / max(tp + fp + fn, 1),
        "mean_confidence": float(confidence.mean()) if len(confidence) else None,
        "mean_latency_ms": float(output["latencies"].mean()),
        "median_latency_ms": float(np.median(output["latencies"])),
        "p95_latency_ms": float(np.percentile(output["latencies"], 95)),
        "throughput_fps": len(val_images) * 1000 / output["latencies"].sum(),
        "TP": int(tp), "FP": int(fp), "FN": int(fn),
    }

deployment_metrics = pd.DataFrame([summarize_validation(o) for o in validation_outputs.values()])
deployment_metrics.to_csv(WORK / "deployment_metrics.csv", index=False)
display(deployment_metrics)

### Estimate the composite score on validation (proxy only)

This applies the published score formula to the notebook's own validation AP, median latency and aggregate throughput. It is useful for **comparing experiments consistently**, but it is not an official score: the lightweight validation evaluator above does not fully reproduce the host's VisDrone ignore-region metric, and the test split differs from validation.


In [ ]:
SPEED_REFERENCE = {
    "onnx_cpu": {"latency_ms": 15.777892, "fps": 60.337638},
    "tensorrt_t4": {"latency_ms": 6.077187, "fps": 162.532436},
}

# Make this cell safe to rerun: remove any metrics generated by a previous run.
proxy_columns = [
    "latency_factor", "throughput_factor", "efficiency",
    "validation_score_proxy", "combined_validation_score_proxy",
]
deployment_metrics = deployment_metrics.drop(
    columns=[c for c in proxy_columns if c in deployment_metrics.columns],
    errors="ignore",
)

# Force the input columns to numeric in case the table came from a CSV/string column.
for column in ["AP50_95", "median_latency_ms", "throughput_fps"]:
    deployment_metrics[column] = pd.to_numeric(deployment_metrics[column], errors="raise")

def validation_score_proxy(row):
    runtime = str(row["runtime"])
    if runtime not in SPEED_REFERENCE:
        raise ValueError(f"Unexpected runtime in validation metrics: {runtime!r}")
    reference = SPEED_REFERENCE[runtime]
    latency_factor = reference["latency_ms"] / (reference["latency_ms"] + float(row["median_latency_ms"]))
    throughput = float(row["throughput_fps"])
    throughput_factor = throughput / (reference["fps"] + throughput)
    efficiency = float(np.sqrt(latency_factor * throughput_factor))
    accuracy = float(np.clip(float(row["AP50_95"]), 0.0, 1.0))
    return pd.Series({
        "latency_factor": latency_factor,
        "throughput_factor": throughput_factor,
        "efficiency": efficiency,
        "validation_score_proxy": accuracy ** 0.70 * efficiency ** 0.30,
    })

proxy = deployment_metrics.apply(validation_score_proxy, axis=1)
deployment_metrics = pd.concat([deployment_metrics, proxy], axis=1)
score_by_runtime = deployment_metrics.set_index("runtime")["validation_score_proxy"].astype(float)
required_runtimes = {"onnx_cpu", "tensorrt_t4"}
missing_runtimes = required_runtimes.difference(score_by_runtime.index.astype(str))
if missing_runtimes:
    raise ValueError(f"Missing validation results for runtime(s): {sorted(missing_runtimes)}")

deployment_metrics["combined_validation_score_proxy"] = float(np.sqrt(
    float(score_by_runtime.loc["onnx_cpu"]) * float(score_by_runtime.loc["tensorrt_t4"])
))
deployment_metrics.to_csv(WORK / "deployment_metrics.csv", index=False)
display(deployment_metrics[["runtime", "AP50_95", "median_latency_ms", "throughput_fps",
                            "efficiency", "validation_score_proxy", "combined_validation_score_proxy"]])


### Generate a confusion matrix for each deployment

In [ ]:
for name, output in validation_outputs.items():
    figure, axis = plt.subplots(figsize=(10, 9))
    matrix = output["confusion"]
    image = axis.imshow(matrix, cmap="Blues")
    labels = CLASS_NAMES + ["background"]
    axis.set_xticks(range(11), labels, rotation=65, ha="right")
    axis.set_yticks(range(11), labels)
    axis.set_xlabel("Predicted category")
    axis.set_ylabel("True category")
    axis.set_title(f"{name}: validation confusion matrix (conf ≥ 0.25, IoU ≥ 0.50)")
    for row, column in zip(*np.nonzero(matrix)):
        axis.text(column, row, str(matrix[row, column]), ha="center", va="center", fontsize=8)
    figure.colorbar(image, ax=axis)
    figure.tight_layout()
    figure.savefig(WORK / f"{name}_confusion_matrix.png", dpi=150)
    plt.show()

### Read the confusion matrix
Rows show the true category and columns show the predicted category. Diagonal counts are correct category matches. The last **background column** counts labeled objects with no accepted matching detection; the last **background row** counts extra detections. Background is a bookkeeping category, not an explicit class predicted by this detector.

If almost every count appears in the last column, the detector found few or no accepted matches at confidence 0.25 and IoU 0.50. A SMOKE run with one epoch and 128 training images can be very weak and can produce zero true positives. Run FULL and examine validation results before judging detector quality. AP uses detections down to confidence 0.001, so AP can be nonzero while the fixed-confidence precision and recall are zero. Confidence is an uncalibrated model score.

## 13. Serialize detections in competition format
Each frame stores a JSON array. Categories return to original VisDrone IDs 1–10. Boxes are original image **pixel xyxy**, not normalized YOLO labels. An image with no detections stores `[]`. Confidence is a model score, not a calibrated probability or an accuracy metric.

In [ ]:
def detection_json(detections, image_shape):
    height, width = image_shape[:2]
    objects = []
    for x1, y1, x2, y2, confidence, class_id in detections:
        box = clip_box([x1, y1, x2, y2], width, height)
        if box[2] <= box[0] or box[3] <= box[1]:
            continue
        objects.append({"class_id": int(class_id) + 1, "bbox": box,
                        "confidence": float(np.clip(confidence, 0, 1))})
    return json.dumps(objects, separators=(",", ":"), allow_nan=False)

### Process all test frames in each deployment
Per-video throughput is frame count divided by the sum of timed processing seconds for that video. Repeat that video's FPS in every row belonging to it. Both runtimes process identical decoded inputs. Full CPU inference on 6,635 frames may take time; wait for both progress bars to finish.

In [ ]:
submission = pd.DataFrame({"frame_id": test_ids})
frame_shapes = {}
for name, (predict, use_cuda) in runtimes.items():
    json_predictions, latency_values = [], []
    video_elapsed_ms = defaultdict(float)
    video_frames = defaultdict(int)
    for frame_id, image_path in tqdm(list(zip(test_ids, test_paths)), desc=f"Test {name}"):
        image = cv2.imread(str(image_path))
        assert image is not None, f"Cannot decode {image_path}"
        detections, latency_ms = timed_prediction(predict, image, use_cuda)
        json_predictions.append(detection_json(detections, image.shape))
        latency_values.append(latency_ms)
        sequence = frame_id.split("/")[0]
        video_elapsed_ms[sequence] += latency_ms
        video_frames[sequence] += 1
        frame_shapes[frame_id] = image.shape[:2]
    submission[f"{name}_detections_json"] = json_predictions
    submission[f"{name}_latency_ms"] = latency_values
    submission[f"{name}_throughput_fps"] = [
        video_frames[f.split("/")[0]] * 1000 / video_elapsed_ms[f.split("/")[0]] for f in test_ids
    ]

## 14. Validate the submission before saving
Accuracy columns do not belong in the participant CSV: accuracy must be calculated from the host's ground truth and submitted detections. We check IDs, column order, finite positive timings, category IDs, confidence and image boundaries.

In [ ]:
def validate_submission(table, expected_ids, shapes):
    assert list(table.columns) == SUBMISSION_COLUMNS
    assert table.frame_id.tolist() == expected_ids and table.frame_id.is_unique
    assert not table.isna().any().any()
    for name in ("onnx_cpu", "tensorrt_t4"):
        timings = table[[f"{name}_latency_ms", f"{name}_throughput_fps"]].to_numpy(float)
        assert np.isfinite(timings).all() and (timings > 0).all()
        video_ids = table.frame_id.str.split("/").str[0]
        assert table.groupby(video_ids)[f"{name}_throughput_fps"].nunique().eq(1).all()
        for frame_id, text in zip(table.frame_id, table[f"{name}_detections_json"]):
            objects = json.loads(text)
            assert isinstance(objects, list)
            height, width = shapes[frame_id]
            for obj in objects:
                assert set(obj) == {"class_id", "bbox", "confidence"}
                assert isinstance(obj["class_id"], int) and 1 <= obj["class_id"] <= 10
                assert 0 <= obj["confidence"] <= 1
                box = np.asarray(obj["bbox"], dtype=float)
                assert box.shape == (4,) and np.isfinite(box).all()
                x1, y1, x2, y2 = box
                assert 0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height
    return True

submission = submission[SUBMISSION_COLUMNS]
assert validate_submission(submission, test_ids, frame_shapes)
submission_path = WORK / "submission.csv"
submission.to_csv(submission_path, index=False)
reloaded = pd.read_csv(submission_path, dtype={"frame_id": str})
assert validate_submission(reloaded, test_ids, frame_shapes)
print(f"Validated {len(reloaded):,} rows: {submission_path}")
display(reloaded.head(2))
# Provide BOTH paths, since some Kaggle submission dialogs expect root submission.csv.
root_submission = Path("/kaggle/working/submission.csv")
if Path("/kaggle/working").exists():
    shutil.copy2(submission_path, root_submission)
    assert root_submission.is_file()
    assert root_submission.stat().st_size == submission_path.stat().st_size
    print("Root-level submission also saved:", root_submission)


### Root submission copy — prevents missing `submission.csv` during upload

The main pipeline writes `edgevision/submission.csv`. Some submission routes default to root `submission.csv`. This cell safely writes a second identical copy and checks both. Run this after the official validation cell and before saving the notebook output.

In [ ]:
from pathlib import Path
import shutil

main_submission = WORK / "submission.csv"
root_submission = Path("/kaggle/working/submission.csv")
assert main_submission.is_file(), f"Missing submission: {main_submission}"
assert len(pd.read_csv(main_submission)) == 6635, "Expected 6,635 rows"
shutil.copy2(main_submission, root_submission)
assert root_submission.is_file()
assert main_submission.read_bytes() == root_submission.read_bytes()
print("Ready to submit:", root_submission)
print("Alternative notebook output:", main_submission)


### Show actual full-test benchmark results

In [ ]:
test_speed = []
for name in runtimes:
    latency = submission[f"{name}_latency_ms"].to_numpy()
    test_speed.append({"runtime": name, "frames": len(submission),
                       "mean_latency_ms": latency.mean(), "median_latency_ms": np.median(latency),
                       "p95_latency_ms": np.percentile(latency, 95),
                       "overall_throughput_fps": len(submission) * 1000 / latency.sum()})
test_speed = pd.DataFrame(test_speed)
test_speed.to_csv(WORK / "test_speed.csv", index=False)
display(test_speed)

## 15. Save the environment and experiment settings
Keep these records with the notebook and exported models so the host can reproduce your timings. Installing the latest release means later runs can use different versions; this record captures the actual run.

In [ ]:
package_names = ["ultralytics", "torch", "onnx", "onnxruntime", "tensorrt-cu12", "numpy"]
cpu_model = platform.processor()
if Path("/proc/cpuinfo").exists():
    cpu_lines = Path("/proc/cpuinfo").read_text().splitlines()
    cpu_model = next((line.split(":", 1)[1].strip() for line in cpu_lines if line.startswith("model name")), cpu_model)
runtime_metadata = {
    "mode": MODE, "model": MODEL_NAME, "best_weights": str(best_weights),
    "epochs_requested": EPOCHS, "epochs_completed": int(getattr(trainer.trainer, "epoch", -1)) + 1,
    "training_patience": PATIENCE, "training_batch": BATCH_SIZE, "workers": WORKERS,
    "python": platform.python_version(), "platform": platform.platform(),
    "packages": {name: metadata.version(name) for name in package_names},
    "cpu": cpu_model, "cpu_threads": CPU_THREADS, "gpu": GPU_NAME,
    "cuda": torch.version.cuda, "tensorrt": trt.__version__, "onnx_precision": "FP32", "tensorrt_precision": "FP16" if TRT_FP16 else "FP32",
    "onnx_providers": onnx_session.get_providers(), "imgsz": IMGSZ, "batch": 1,
    "confidence_threshold": CONF, "diagnostic_confidence": DIAGNOSTIC_CONF,
    "nms_iou": NMS_IOU, "max_det": MAX_DET, "seed": SEED,
    "train_frames": len(train_images), "validation_frames": len(val_images), "test_frames": len(test_ids),
    "timing_scope": "decoded image -> preprocessing -> inference -> NMS -> original-pixel CPU detections; CUDA synchronized",
    "video_throughput": "frames / summed timed prediction seconds for each sequence",
    "source_roots": {"train": str(train_source), "validation": str(val_source), "test": str(test_source)},
}
(WORK / "runtime_metadata.json").write_text(json.dumps(runtime_metadata, indent=2))
print("Artifacts saved in", WORK)

### Check the publication boundary
The output folder contains model files, metrics, the submission and confusion matrices. Prepared dataset images and labels stay in temporary storage. Public notebooks must not redistribute restricted dataset frames or annotation files.

In [ ]:
assert DATASET_WORK.resolve().is_relative_to(TEMP_ROOT.resolve())
assert not DATASET_WORK.resolve().is_relative_to(WORK.resolve())
restricted_image_outputs = [p for p in WORK.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".webp"}]
assert not restricted_image_outputs, f"Move restricted frame outputs out of published artifacts: {restricted_image_outputs[:5]}"
assert not list(WORK.rglob("labels")), "Prepared annotation folders must stay in temporary storage."
print("Publication check passed: no prepared dataset frames or label folders in outputs.")

## 16. Submit on Kaggle
The competition's calibrated score balances accuracy and speed in the range 0–1. For each deployment, let `A` be the scorer's AP50–95, `L` its median frame latency in milliseconds, and `F` its aggregate video throughput:

`latency_factor = L_ref / (L_ref + L)`  
`throughput_factor = F / (F_ref + F)`  
`efficiency = sqrt(latency_factor * throughput_factor)`  
`deployment_score = A**0.70 * efficiency**0.30`  
`final_score = sqrt(onnx_cpu_score * tensorrt_t4_score)`

The measured reference values are CPU **15.777892 ms / 60.337638 FPS** and T4 **6.077187 ms / 162.532436 FPS**. They come from the optional YOLO11n reference at image size 320, FP32, batch 1 and two CPU threads. Faster inference continues improving the bounded speed factors; it is not clipped at the reference speed. These constants describe speed normalization, not mandatory model choices or an accuracy target.

The host's genuine saved Version 3 submission passed the Kaggle sandbox with public **0.00005** and private **0.00011** under the earlier Version 4 scorer. Those are weak one-epoch SMOKE accuracy results, not expected FULL model quality. Metric Version 5 passed Kaggle competition-metric validation and was attached on 6 October 2026. Its continuous speed formula and measured reference values below are now active. The same genuine saved submission also passed Version 5, with public **0.00004** and private **0.00009**. The earlier Version 4 scores above are historical smoke results.

1. Finish the notebook run and confirm the validation cell reports **6,635 rows**.
2. After a successful interactive run, choose **Save Version → Quick Save**. Open **Advanced Settings**, enable **Save output for this version**, and save. Verify that the saved version's Output tab contains `edgevision/submission.csv`. Alternatively, **Save & Run All** reruns the notebook in a fresh session; wait for success and check its outputs before submitting. In the host's test, two fresh saved runs failed during Kaggle's dataset mounting before any notebook code ran; the complete interactive run succeeded. A version number alone does not establish a successful run.
3. In the competition, choose **Submit Predictions**, select the **Notebook** submission option, choose the saved notebook version and its `edgevision/submission.csv`, add a short description, and submit. This output-selection path was exercised in the host's pipeline test. If file upload is available instead, download the CSV from the saved version's Output tab and upload it. Keep generated models, metrics and the environment record for review.
4. Wait for processing and check **My Submissions** for an accepted score or a specific error. The leaderboard shows the competition's composite score; it does not show every diagnostic metric as a separate column. Every new submission must be checked for successful scoring even though the host's genuine 6,635-row pipeline test was accepted.
5. Before the deadline, select your final submission(s) according to the host's stated limit. Keep your reproducible notebook available for host review and publish it after the competition under the host's instructions. Restricted competition data must remain subject to its permissions.

**Recommended experiment order:** (1) run SMOKE to verify the pipeline; (2) run FULL with `IMGSZ=320`, `TRT_FP16=False`, and `CPU_THREADS=2` and retain the metrics; (3) change only `IMGSZ=640`, rerun from configuration onward, and compare the combined validation-score proxy; (4) after selecting image size, toggle only `TRT_FP16=True` and rerun the export/deployment/evaluation/submission cells, checking parity and the score proxy again. Keep actual metadata and versioned outputs for each experiment. Do not assume 640px or FP16 must win. To explore a larger detector, change `MODEL_NAME` as a separate experiment and ensure the ONNX decoder is compatible with its output.

If a conversion fails, resolve the first failing cell and rerun subsequent cells. Never replace a failed runtime's values with estimates, PyTorch values or another device's measurements.

References: [Ultralytics export documentation](https://docs.ultralytics.com/modes/export/), [NVIDIA TensorRT pip installation](https://docs.nvidia.com/deeplearning/tensorrt/latest/installing-tensorrt/install-pip.html), [VisDrone dataset](https://github.com/VisDrone/VisDrone-Dataset), [competition](https://www.kaggle.com/competitions/edgevision).